In [1]:
import os    # путь к файлу
import zstandard as zstd    # декомпрессор
import tarfile    # распаковка архива
import glob    # поиск файлов по шаблону

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from joblib import Parallel, delayed, cpu_count    # распараллеливание

import math

import warnings

In [2]:
# Устанавливаем переменную окружения, чтобы игнорировать все предупреждения.
# Если вы хотите, чтобы ваши собственные предупреждения отображались,
# но, например, игнорировались DeprecationWarning, вы можете настроить фильтр.
# Формат: "действие:сообщение:категория"
os.environ['PYTHONWARNINGS'] = 'default' # Это значение по умолчанию, предупреждения будут показаны.

In [3]:
path = '.'

# Вспомогательные функции

### Названия файлов

In [4]:
def create_filename_2d(dt, level):
    """
    Создает название файла по datetime и уровню

    Parameters:
    dt (datetime): объект datetime
    level (int): уровень (например, 1000, 850, 500)

    Returns:
    str: название файла в формате mask_cycl_level_year_month_day_hour.npy
    """
    # Форматируем компоненты даты с ведущими нулями где необходимо
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    # Создаем название файла
    filename = f"mask_cycl_{level}_{year}_{month:02d}_{day:02d}_{hour:02d}.npy"

    return filename

In [5]:
def create_filename_3d(dt):
    """
    Создает название файла по datetime

    Parameters:
    dt (datetime): объект datetime

    Returns:
    str: название файла в формате mask_cycl_year_month_day_hour.npy
    """
    # Форматируем компоненты даты с ведущими нулями где необходимо
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    # Создаем название файла
    filename = f"mask_cycl_{year}_{month:02d}_{day:02d}_{hour:02d}.npy"

    return filename

### Маски

In [6]:
def create_bit_masks(arr):
    """
    Самая быстрая версия - создаем все маски сразу.
    """
    unique_values = np.unique(arr)
    return {val: (arr == val).astype(np.uint8) for val in unique_values if val != 0}

In [7]:
def create_2d_mask(bit_masks_dict, reference_shape=(37, 144)):
    """
    Создает 2D маску. Если объектов нет, возвращает нулевой массив заданного размера.
    """
    if not bit_masks_dict:
        return np.zeros(reference_shape, dtype=np.int32)
    
    # Используем форму первой маски из словаря
    first_mask = next(iter(bit_masks_dict.values()))
    result_mask = np.zeros_like(first_mask, dtype=np.int32)
    
    for key, bit_mask in bit_masks_dict.items():
        result_mask += bit_mask.astype(np.int32) * key
    
    return result_mask

### Поправка на сетку

In [8]:
def band_area_ratios(latitudes, degrees=True):

    if degrees:
        latitudes = [math.radians(x) for x in latitudes]

    areas = [
        math.sin(latitudes[i+1]) - math.sin(latitudes[i])
        for i in range(len(latitudes)-1)
    ]

    polar_area = areas[-1]

    return [a / polar_area for a in areas]

In [9]:
# границы полос
lats = []
lats.append(0.0)
for i in range(125, 9000, 250):
    lats.append(i / 100)
lats.append(90.0)
# print(lats)
# print(len(lats))

In [10]:
ratios = band_area_ratios(lats)
# print(ratios)
# print(len(ratios))

In [11]:
ratios_vector = np.array(ratios[::-1])
# ratios_vector

In [12]:
ratios_matrix = np.tile(ratios_vector.reshape(-1, 1), (1, 144))
# ratios_matrix

# Преобразование масок

In [13]:
levels = [1000, 925, 850, 700, 600, 500, 300]    #### [300, 500, 600, 700, 850, 925, 1000]

In [14]:
start_time = datetime(1979, 1, 1, 0, 0)    #### datetime(1979, 1, 1, 0, 0)
end_time = datetime(2023, 12, 31, 18, 0)    #### datetime(2023, 12, 31, 18, 0)
time_delta = timedelta(hours=6)

times = []
t = start_time
while t <= end_time:
    times.append(t)
    t += time_delta

In [15]:
def process_datetime_ver2(t):
    try:
        # Константа размера сетки
        GRID_SHAPE = (37, 144)
        
        l_bottom = levels[0]
        # Загружаем базовый уровень
        mask_bottom_full = np.load(path + '/mask_cycl_2d' + '/' + create_filename_2d(t, l_bottom))
        mask_bottom = mask_bottom_full[0]
        
        # Проверка размера базового слоя
        if mask_bottom.shape != GRID_SHAPE:
            print(f"Skip {t}: Base level {l_bottom} has wrong shape {mask_bottom.shape}")
            # return None

            warnings.warn(
                f"Timestamp {t}: Base level {l_bottom} has wrong shape {mask_bottom.shape}. Expected {GRID_SHAPE}. Skipping.",
                RuntimeWarning
            )
            return None

        bit_masks_bottom = create_bit_masks(mask_bottom)
        res_3d_mask = mask_bottom_full 

        # Инициализация сквозного счетчика ID для этого момента времени
        if bit_masks_bottom:
            current_max_id = max(bit_masks_bottom.keys())
        else:
            current_max_id = 0

        for i in range(1, len(levels)):
            l_top = levels[i]
            mask_top_loaded = np.load(path + '/mask_cycl_2d' + '/' + create_filename_2d(t, l_top))[0]
            
            # Проверка размера текущего слоя
            if mask_top_loaded.shape != GRID_SHAPE:
                print(f"Skip {t}: Level {l_top} has wrong shape {mask_top_loaded.shape}. Expected {GRID_SHAPE}")
                # return None

                warnings.warn(
                    f"Timestamp {t}: Level {l_top} has wrong shape {mask_top_loaded.shape}. Expected {GRID_SHAPE}. Skipping this timestamp.",
                    RuntimeWarning
                )
                return None

            bottom_unique_keys = sorted(bit_masks_bottom.keys())
            top_unique_keys = sorted(int(k) for k in np.unique(mask_top_loaded) if k != 0)

            new_bit_masks_top = {}

            if top_unique_keys:
                # Расчет пересечений
                intersection_df = pd.DataFrame(index=bottom_unique_keys, columns=top_unique_keys, dtype=float)
                for bottom_key in bottom_unique_keys:
                    mask_intersection = np.multiply(bit_masks_bottom.get(bottom_key), mask_top_loaded)
                    bit_masks_intersection = create_bit_masks(mask_intersection)
                    
                    for intersection_key in bit_masks_intersection.keys():
                        intersection_res = np.sum(bit_masks_intersection.get(intersection_key) * ratios_matrix)
                        intersection_df.loc[bottom_key, intersection_key] = intersection_res

                # Словарь потенциальных переименований
                rename_options = {top_key: [] for top_key in top_unique_keys}
                for bottom_key in bottom_unique_keys:
                    row = intersection_df.loc[bottom_key]
                    non_zero_values = row[row > 0]
                    if len(non_zero_values) > 0:
                        max_val = non_zero_values.max()
                        max_cols = non_zero_values[non_zero_values == max_val].index
                        if len(max_cols) == 1:
                            rename_options[max_cols[0]].append(bottom_key)

                # Присвоение имен объектам на верхнем уровне
                for top_key in top_unique_keys:
                    possible_renames = rename_options[top_key]
                    
                    if len(possible_renames) == 0:
                        current_max_id += 1
                        new_name = current_max_id
                    elif len(possible_renames) == 1:
                        new_name = possible_renames[0]
                        current_max_id = max(current_max_id, new_name)
                    else:
                        # Слияние: выбираем по максимальной площади из таблицы пересечений
                        new_name = intersection_df[top_key].idxmax()
                        current_max_id = max(current_max_id, new_name)
                    
                    new_bit_masks_top[new_name] = (mask_top_loaded == top_key).astype(np.uint8)

            # Создаем 2D маску уровня (если объектов нет, вернет массив нулей 37x144)
            new_mask_top = create_2d_mask(new_bit_masks_top, reference_shape=GRID_SHAPE)
            
            # Подготовка к следующему уровню
            bit_masks_bottom = new_bit_masks_top
            
            # Добавляем в 3D стек
            new_mask_top_expanded = np.expand_dims(new_mask_top, axis=0)
            res_3d_mask = np.concatenate((res_3d_mask, new_mask_top_expanded), axis=0)

        # Сохранение итогового файла
        save_path = path + '/mask_cycl_3d' + '/' + create_filename_3d(t)
        np.save(save_path, res_3d_mask)
        return t

    except Exception as e:
        print(f"Error processing timestamp {t}: {e}")
        return None

### С распараллеливанием

In [16]:
n_jobs = cpu_count()  # оставить 1 ядро системе
print(n_jobs)

8


In [17]:
start = datetime.now()
print(start)

Parallel(n_jobs=n_jobs)(delayed(process_datetime_ver2)(t) for t in times)

finish = datetime.now()
print(finish)

programm_length = (finish - start).total_seconds()
print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

2026-04-09 17:42:48.585435
2026-04-09 18:21:18.573652
0.0 hours 38.0 minutes 29.988217000000077 seconds
